In [1]:
# --- Setup and Configuration ---
import nest_asyncio
import os
import re
import ast

# Apply nest_asyncio for running asyncio in environments like Jupyter/IPython
nest_asyncio.apply()

# --- Data Handling and Vectorization ---
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import Dataset, load_dataset, load_from_disk
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer

# --- Vector Database Client (Qdrant) ---
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams,
    Distance,
    Filter,
    FieldCondition,
    MatchValue
)

# --- Model Imports ---
# Assuming BankingClassifier is a custom class defined in this module path
from artifacts.intent_classifier import BankingClassifier

# --- LLM and Ragas Evaluation Imports ---
# For LangChain/OpenAI integration with Ragas
from langchain_openai import ChatOpenAI
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig
from ragas import evaluate
from ragas.metrics import (
    context_recall,
    context_precision,
    answer_correctness,
    answer_similarity
)


from fastembed import SparseTextEmbedding

# Initialize your sparse encoder (do this once outside the function)
# sparse_encoder = SparseTextEmbedding(model_name="prithivida/Splade_PP_en_v1")

# --- Utility/Serialization ---
import joblib

C:\Users\vncpyy7h\AppData\Local\Temp\ipykernel_12500\3224888460.py:39: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
C:\Users\vncpyy7h\AppData\Local\Temp\ipykernel_12500\3224888460.py:39: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\vncpyy7h\AppData\Local\Temp\ipykernel_12500\3224888460.py:39: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
C:\Users\vncpyy7h\AppData\

In [2]:
pd.set_option('display.max_colwidth', None)

## Retrieval and integrate with LLM 

In [ ]:

# --- Configuration ---
QDRANT_URL = "your_qdrant_url"  # Replace with your actual Qdrant URL
QDRANT_API_KEY = "your_qdrant_api_key"  # Replace with your actual Qdrant API key
COLLECTION_NAME = "banking_faq"
OPENAI_API_KEY = "your_openai_api_key"  # Replace with your actual OpenAI API key
MODEL_DIR = "./"

# Initialize Clients
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
# Initialize your BankingClassifier class (from your previous code)
bot = BankingClassifier(model_dir=MODEL_DIR)
bot.load() # This loads the encoder and intent classifier

# Initialize OpenAI
openai_client = OpenAI(api_key=OPENAI_API_KEY)
# 1. Set your OpenAI API key for Ragas
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

--- Initializing Classifier ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading complete in 10.91 seconds.


In [4]:
sparse_encoder = SparseTextEmbedding(model_name="prithivida/Splade_PP_en_v1")

In [ ]:
from qdrant_client import models

def get_banking_answer(user_query, use_hybrid=False):

    # 1. Blocklist Guardrail (Fast, low-cost)
    RISK_KEYWORDS = [r"invest", r"crypto", r"bitcoin", r"hack", r"launder"]
    if any(re.search(word, user_query.lower()) for word in RISK_KEYWORDS):
        return {"answer": "I cannot assist with that request as it falls outside my banking policy.", "category": "BLOCKED"}
    # 1. Classify the CATEGORY
    category, conf, latency = bot.predict(user_query)
    
    # 2. Embed the query
    query_vector = bot.encoder.encode(user_query).tolist()
    sparse_vector = None

    # Define the Category Filter
    filter_condition = models.Filter(
        must=[models.FieldCondition(key="category", match=models.MatchValue(value=category))]
    )

    # --- HYBRID SEARCH LOGIC ---
    if use_hybrid:
        # Generate Sparse (Keyword) Vector
        sparse_vector = list(sparse_encoder.embed(user_query))[0] # Example for fastembed
        # For this example, let's assume sparse_vector is ready
        
        prefetch = [
            models.Prefetch(query=query_vector, using="dense", limit=10, filter=filter_condition),
            models.Prefetch(query=sparse_vector, using="sparse", limit=10, filter=filter_condition)
        ]
        
        search_results = client.query_points(
            collection_name="banking_faq",
            prefetch=prefetch,
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=1
        )
    
    # --- PURE VECTOR SEARCH LOGIC (Original) ---
    else:
        search_results = client.query_points(
            collection_name="banking_faq",
            query=query_vector,
            query_filter=filter_condition,
            limit=1
        )
    
    # 3. Handle Results
    if search_results.points:
        best_match = search_results.points[0].payload
        return {
            "answer": best_match['answer'],
            "category": category,
            "confidence": conf
        }
    else:
        # Fallback Logic (Repeat same process but without filter)
        print("No match in category, searching entire database...")
        
        if use_hybrid:
            prefetch = [
                models.Prefetch(query=query_vector, using="dense", limit=10),
                models.Prefetch(query=sparse_vector, using="sparse", limit=10)
            ]
            fallback_results = client.query_points(
                collection_name="banking_faq",
                prefetch=prefetch,
                query=models.FusionQuery(fusion=models.Fusion.RRF),
                limit=1
            )
        else:
            fallback_results = client.query_points(
                collection_name="banking_faq",
                query=query_vector,
                limit=1
            )
            
        if fallback_results.points:
            return {
                "answer": fallback_results.points[0].payload['answer'],
                "category": "GENERAL"
            }
        else:
            return {"answer": "I'm sorry, I don't know.", "category": "GENERAL"}

In [6]:
get_banking_answer('I got to activate a card , can you help me?')

{'answer': 'I\'m happy to help! I\'d be happy to assist you with activating your card. To get started, please follow these steps:\n\n1. Download and install our mobile app from the App Store or Google Play Store.\n2. Open the app and sign in to your account.\n3. Navigate to the "Cards" section or a similar option.\n4. Look for the option to activate a new card.\n5. Enter the necessary information such as the card number, expiration date, and CVV.\n6. Follow the on-screen prompts to complete the activation process.\n\nIf you encounter any difficulties during the activation process or have any questions, please don\'t hesitate to reach out to our customer support team at {{Customer Support Phone Number}}. They\'re available {{Customer Support Working Hours}} to assist you further. Enjoy using your activated card!',
 'category': 'CARD',
 'confidence': 0.9999811830568314}

In [8]:
def get_banking_answer_with_llm(user_query):
    # 1. Retrieve the data (using your existing logic)
    data = get_banking_answer(user_query) # This returns the dict with 'answer' and 'category'
    
    if data['answer'] == "I'm sorry, I don't know.":
        return data['answer']
    #print(data['answer'])
    # 2. Construct the Prompt for GPT-4o-mini
    prompt = f"""
    You are a banking support assistant. 
    Your ONLY source of information is the "KNOWLEDGE BASE" provided below. 
    
    RULES:
    1. STRICT ADHERENCE: You must use the steps provided in the KNOWLEDGE BASE exactly as they appear. 
    2. NO FABRICATION: Do not add steps, do not remove steps, and do not change the terminology (e.g., if it says 'Forgot Password', do not say 'Sign In').
    3. FORMATTING: Output the response in a clear, numbered list format.
    4. ALIGNMENT: If the KNOWLEDGE BASE is missing a step, do not invent one. State that the instructions are as provided.

    KNOWLEDGE BASE:
    {data['answer']}

    USER QUESTION:
    {user_query}

    RESPONSE:
    """

    # 3. Call GPT-4o-mini
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful banking assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1 # Lower temperature = more factual, less creative
    )

    return response.choices[0].message.content



In [9]:
# --- Test it ---
final_answer = get_banking_answer_with_llm("I make a mistake, how could I cancel a bank transfer?")
print(final_answer)

1. First, log in to your online banking account or mobile banking app.

2. Navigate to the section that displays your recent transactions or transfers.

3. Locate the specific bank transfer that you want to cancel. Look for options such as "Cancel," "Reverse," or "Undo" next to the transaction.

4. If you don't see any direct cancellation options, you can contact your bank's customer support service through their hotline or online chat. They will guide you through the cancellation process and provide any additional steps or requirements.

5. It's important to act as soon as possible to increase the chances of successfully canceling the transfer. Transfers that have already been processed may be more difficult to cancel.

Remember, every bank may have slightly different procedures, so it's best to follow the instructions provided by your specific bank. If you have any further questions or need additional assistance, please let me know.


### Groundedness evaluation
- The reponse should have enough information. For banking products, missing or adding things are unacceptable  
- It checks results from vector database retrieval

In [10]:
#dataset = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset", split="train")

In [12]:
# 3. Initialize the stronger evaluator LLM (gpt-4o-mini is highly recommended for Ragas)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))

# 4. Load the specific retail banking dataset
print("Loading banking dataset from Hugging Face...")
raw_dataset = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset", split="train")

# 5. Select a small subset to evaluate (e.g., 50 rows) to manage costs and speed
EVAL_SAMPLE_SIZE = 100
eval_subset = raw_dataset.shuffle(seed=1).select(range(EVAL_SAMPLE_SIZE))

# 6. Lists to collect data for Ragas
questions = []
contexts = []      # Must be a list of lists of strings
ground_truths = []

# Define the regex pattern to match anything inside {{ }}
placeholder_pattern = r'\{\{.*?\}\}'


def unwrap_placeholders(text):
    """
    Dynamically removes the double curly braces {{ and }} but keeps the 
    inner text intact. This maintains grammatical completeness with zero hardcoding.
    """
    if not isinstance(text, str):
        return text
    return re.sub(r'\{\{(.*?)\}\}', r'\1', text)


print(f"Starting retrieval process for {EVAL_SAMPLE_SIZE} samples...")

# ==========================================
# Step 1: The Retrieval Loop (Populates & Cleans the lists)
# ==========================================
for index, row in enumerate(eval_subset):
    query = row["instruction"]      # The user's question
    expected_truth = row["response"] # The gold standard answer
    
    print(f"[{index + 1}/{EVAL_SAMPLE_SIZE}] Retrieving for: '{query[:50]}...'")
    
    # Run your Qdrant retrieval function
    try:
        retrieved_data = get_banking_answer(query)
        retrieved_context = retrieved_data["answer"]
        
        # --- Context Cleaning Logic ---
        if isinstance(retrieved_context, str):
            # A. Clean string-wrapped lists: "['text']" -> "text"
            if retrieved_context.startswith("['") and retrieved_context.endswith("']"):
                try:
                    parsed_list = ast.literal_eval(retrieved_context)
                    if isinstance(parsed_list, list) and len(parsed_list) > 0:
                        retrieved_context = parsed_list[0]
                except Exception:
                    retrieved_context = retrieved_context.strip("[]'\"").replace("\\'", "'")
            
            # B. Clean literal backslash escapes
            retrieved_context = retrieved_context.replace('\\n', '\n').replace('""', '"').strip()
            expected_truth = expected_truth.strip()
            
            # C. DYNAMIC UNWRAPPING: Keep the noun, strip only the braces
            retrieved_context = unwrap_placeholders(retrieved_context)
            expected_truth = unwrap_placeholders(expected_truth)
            
            # D. Clean up any accidental double spaces left behind
            retrieved_context = re.sub(r'\s+', ' ', retrieved_context).strip()
            expected_truth = re.sub(r'\s+', ' ', expected_truth).strip()
                
    except Exception as e:
        print(f"Error retrieving for query '{query}': {e}")
        retrieved_context = "Error during retrieval"

    # Append to our evaluation datasets
    questions.append(query)
    contexts.append([retrieved_context]) 
    ground_truths.append(expected_truth)


# ==========================================
# Step 2: Batch Ragas Evaluation
# ==========================================
print(f"\nInitiating Batch Ragas evaluation for {EVAL_SAMPLE_SIZE} samples...")

# Construct the full batch dataset
eval_data = {
    "question": questions,
    "contexts": contexts,
    "ground_truth": ground_truths
}
ragas_dataset = Dataset.from_dict(eval_data)

# Configure the run to use 2 parallel workers to prevent rate limits or hanging
config = RunConfig(max_workers=2, timeout=60.0)

try:
    # Run evaluation on the entire batch at once
    results = evaluate(
        dataset=ragas_dataset,
        metrics=[context_recall, context_precision],
        run_config=config,
        llm=evaluator_llm  # Force Ragas to use gpt-4o-mini instead of gpt-3.5-turbo
    )
    
    # --- FINAL RESULTS DISPLAY ---
    
    print("\n=== OVERALL RETRIEVAL AVERAGE SCORES ===")
    print(results)

    # Convert the results to a Pandas DataFrame to view the individual details
    df_results = results.to_pandas()
    
    print("\n=== DETAILED BREAKDOWN PER QUERY ===")
    display(df_results[['user_input', 'retrieved_contexts', 'reference', 'context_recall', 'context_precision']])

except Exception as e:
    print(f"\nBatch evaluation failed: {e}")

C:\Users\vncpyy7h\AppData\Local\Temp\ipykernel_4196\3839465065.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))


Loading banking dataset from Hugging Face...
Starting retrieval process for 100 samples...
[1/100] Retrieving for: 'i cant find my accfount fees i need help checking ...'
[2/100] Retrieving for: 'I have made a mistake and I transferred money to t...'
[3/100] Retrieving for: 'can I see the loan payment?...'
[4/100] Retrieving for: 'i have to see the mortgage payments how could i do...'
[5/100] Retrieving for: 'i cannot pay for my loan help me canceling it...'
[6/100] Retrieving for: 'I need help to see the current balance on my cards...'
[7/100] Retrieving for: 'I want to send money, ia it possible to perform a ...'
[8/100] Retrieving for: 'wanna check the  fucking card annual fee i need as...'
[9/100] Retrieving for: 'i dont have a fucking profile can ya help me creat...'
[10/100] Retrieving for: 'I'm looking for a mortgaeg...'
[11/100] Retrieving for: 'i got to check my fees can uhwlp me...'
[12/100] Retrieving for: 'I can't find my fucking card annual fee, I need as...'
[13/100] Retr

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]


=== OVERALL RETRIEVAL AVERAGE SCORES ===
{'context_recall': 0.9796, 'context_precision': 0.9900}

=== DETAILED BREAKDOWN PER QUERY ===


,user_input,retrieved_contexts,reference,context_recall,context_precision
0,i cant find my accfount fees i need help checking them,"[I'm genuinely sorry for the inconvenience you're experiencing in locating your account fees. I'm here to help you with that. To check your account fees, please follow these steps: 1. Log in to your account on our website. 2. Navigate to the ""Account"" or ""Profile"" section. 3. Look for an option like ""View Account Fees"" or ""Billing Information"". 4. Click on that option to access detailed information about your account fees. If you're unable to locate the account fees or need further assistance, please provide me with your account details or any other relevant information, and I'll gladly assist you further.]","I'm genuinely sorry for the inconvenience you're experiencing in locating your account fees. I'm here to help you with that. To check your account fees, please follow these steps: 1. Log in to your account on our website. 2. Navigate to the ""Account"" or ""Profile"" section. 3. Look for an option like ""View Account Fees"" or ""Billing Information"". 4. Click on that option to access detailed information about your account fees. If you're unable to locate the account fees or need further assistance, please provide me with your account details or any other relevant information, and I'll gladly assist you further.",1.000000,1.0
1,"I have made a mistake and I transferred money to the wrong account, could you help me cancelling a transfer?","[Please accept my apologies for any inconvenience caused by the mistaken bank transfer. I understand how important it is to resolve this issue for you. Rest assured, I'm here to help you cancel the transfer to the wrong account. To proceed with canceling the transfer, we'll need some specific details. Please provide me with the following information: 1. Your Name: So I can verify your account details. 2. Date and time of the transfer: This will assist in locating the transaction. 3. The amount and currency transferred: This helps identify the specific transaction among others. Once I have these details, I'll escalate the issue to our support team, who will take the necessary steps to cancel the transfer. Please note that depending on the circumstances, cancellation may not always be guaranteed. However, we'll do everything we can to assist you. If you have any additional questions or concerns, feel free to let me know. We're here to provide you with the support you need.]","Please accept my apologies for any inconvenience caused by the mistaken bank transfer. I understand how important it is to resolve this issue for you. Rest assured, I'm here to help you cancel the transfer to the wrong account. To proceed with canceling the transfer, we'll need some specific details. Please provide me with the following information: 1. Your Name: So I can verify your account details. 2. Date and time of the transfer: This will assist in locating the transaction. 3. The amount and currency transferred: This helps identify the specific transaction among others. Once I have these details, I'll escalate the issue to our support team, who will take the necessary steps to cancel the transfer. Please note that depending on the circumstances, cancellation may not always be guaranteed. However, we'll do everything we can to assist you. If you have any additional questions or concerns, feel free to let me know. We're here to provide you with the support you need.",1.000000,1.0
2,can I see the loan payment?,"[I'd be happy to assist you in checking your loan payment. To view your loan payment details, you can log in to your account on our website or mobile app. Once you're logged in, navigate to the ""Loan"" or ""Account"" section, where you should be able to find the information you need. If you encounter any issues or have trouble locating the payment details, please let me know, and I'll guide you through the process.]","I'd be happy to assist you in checking your loan pay

In [13]:
df_results.columns

Index(['user_input', 'retrieved_contexts', 'reference', 'context_recall',
       'context_precision'],
      dtype='str')

In [14]:
raw_dataset.filter(lambda example: example['instruction'] == 'show me the loan payments')['response']

Column(['I\'ll do my best! I can help you with that. Checking your loan payments is a breeze. Here\'s what you need to do:\n\n1. Log in to your account on our website or app.\n2. Navigate to the "Loan" section or "My Loans" tab.\n3. Look for the "Payments" or "Payment History" option.\n4. Click on it to access your loan payment details.\n\nThere, you\'ll find a comprehensive record of your loan payments, including the due dates, amounts paid, and any outstanding balances. If you need further assistance or have any questions regarding your loan payments, feel free to reach out to our customer support team at {{Customer Support Phone Number}} or through the Live Chat on our website at {{Company Website URL}}. We\'re here to ensure a smooth loan payment experience for you.'])

In [15]:
df_results[['user_input','retrieved_contexts','reference','context_recall','context_precision']].sort_values(by='context_recall')

,user_input,retrieved_contexts,reference,context_recall,context_precision
70,i want help canceling a loan,"[Certainly, I can assist you with canceling your loan. I understand that canceling a loan can be a significant decision, and I'm here to guide you through the process. To cancel your loan, please follow these steps: 1. Contact our customer support team at Customer Support Phone Number or through our Live Chat service on our website at Company Website URL. They will be able to provide you with the necessary guidance and assist you with canceling your loan. 2. Prepare any relevant information or documentation related to your loan, such as your loan account number or any supporting documents you may have. 3. When contacting our customer support team, clearly explain your intention to cancel the loan. They may ask you for specific details to verify your identity and locate your loan account. 4. Once the cancellation process is initiated, our customer support team will guide you through the next steps, which may include filling out a cancellation form or providing additional information. 5. Take note of any confirmation numbers or reference numbers provided by our customer support team. This will serve as proof of your loan cancellation. If you have any further questions or require additional assistance, please don't hesitate to reach out. We're here to help you throughout the cancellation process.]","I'm sorry to hear that you're looking to cancel a loan. I'll do my best to assist you with that. To cancel your loan, please follow these steps: 1. Contact our customer support team at Customer Support Phone Number or email us at Customer Support Email Address. They will guide you through the cancellation process and provide any necessary forms or documentation. 2. Make sure to have your loan details ready, such as the loan account number, loan amount, and any applicable identification information. 3. Depending on the type of loan, there may be specific terms and conditions for cancellation. Our customer support team will be able to provide you with all the necessary information. 4. It's important to note that there may be fees or penalties associated with loan cancellations. Our customer support team will inform you of any applicable charges. If you have any further questions or need assistance, feel free to reach out to our customer support team. We're here to help make this process as smooth as possible for you.",0.272727,1.0
39,wanna check the <fucking> current balance of my bank account were can i do it,"[Certainly, I can assist you with that! Checking your current balance is an essential part of managing your bank account effectively. Here's how you can do it: 1. Log in to your online banking portal or mobile banking app. 2. Navigate to the ""Account Summary"" or ""Balances"" section. 3. You should be able to see your current account balance displayed there. 4. If you can't find it, look for options like ""My Accounts"" or ""Account Details"" and select the specific account you want to check the balance for. 5. If you're still having trouble finding the balance, reach out to our customer support team at Customer Support Phone Number. They will be more than happy to assist you. Remember, keeping track of your current balance regularly is a great way to stay informed about your financial standing. If you have any further questions or need assistance with anything else, feel free to let me know!]","Certainly, I can assist you with that! Checking your current balance is an essential part of managing your bank account effectively. Here's how you can do it: 1. Log in to your online banking portal or mobile banking app. 2. Navigate to the ""Account Summary"" or ""Balances"" section. 3. You should be able to see your current account balance displayed there. 4. If you can't find it, look for options like ""My Accounts"" or ""Account Details"" and select the specific account you want to check the balance for. 5. If you're still havin

In [16]:
raw_dataset.filter(lambda example: example['instruction'] == 'show me the loan payments')['response']

Column(['I\'ll do my best! I can help you with that. Checking your loan payments is a breeze. Here\'s what you need to do:\n\n1. Log in to your account on our website or app.\n2. Navigate to the "Loan" section or "My Loans" tab.\n3. Look for the "Payments" or "Payment History" option.\n4. Click on it to access your loan payment details.\n\nThere, you\'ll find a comprehensive record of your loan payments, including the due dates, amounts paid, and any outstanding balances. If you need further assistance or have any questions regarding your loan payments, feel free to reach out to our customer support team at {{Customer Support Phone Number}} or through the Live Chat on our website at {{Company Website URL}}. We\'re here to ensure a smooth loan payment experience for you.'])

In [17]:
raw_dataset

Dataset({
    features: ['tags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 25545
})

### Answer correctness / answer similarty 
- Evaluate answers generated from LLM. Make sure it wont response not far away from ground truth 

In [18]:

# 3. Initialize the stronger evaluator LLM
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0.0))

# 4. Load the specific retail banking dataset
print("Loading banking dataset from Hugging Face...")
raw_dataset = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset", split="train")

# 5. Select a small subset to evaluate (e.g., 50 rows) to manage costs and speed
EVAL_SAMPLE_SIZE = 100
eval_subset = raw_dataset.shuffle(seed=1).select(range(EVAL_SAMPLE_SIZE))

# 6. Lists to collect data for Ragas
questions = []
contexts = []      # Must be a list of lists of strings
answers = []       # NEW: Holds the final generated response from your LLM
ground_truths = []

# Define the regex pattern to match anything inside {{ }}
placeholder_pattern = r'\{\{.*?\}\}'


def unwrap_placeholders(text):
    """
    Dynamically removes the double curly braces {{ and }} but keeps the 
    inner text intact to maintain grammatical correctness.
    """
    if not isinstance(text, str):
        return text
    return re.sub(r'\{\{(.*?)\}\}', r'\1', text)


print(f"Starting end-to-end generation process for {EVAL_SAMPLE_SIZE} samples...")

# ==========================================
# Step 1: Run the Full Generation Pipeline
# ==========================================
for index, row in enumerate(eval_subset):
    query = row["instruction"]      # The user's question
    expected_truth = row["response"] # The gold standard answer
    
    print(f"[{index + 1}/{EVAL_SAMPLE_SIZE}] Generating response for: '{query[:50]}...'")
    
    # A. Retrieve context from Qdrant
    try:
        retrieved_data = get_banking_answer(query)
        retrieved_context = retrieved_data["answer"]
    except Exception as e:
        print(f"Error during retrieval: {e}")
        retrieved_context = "Error during retrieval"

    # B. Generate the final LLM response using your full RAG function
    try:
        generated_response = get_banking_answer_with_llm(query)
    except Exception as e:
        print(f"Error during generation: {e}")
        generated_response = "Error during generation"

    # C. Context Cleaning & Unwrapping (for Ragas inputs)
    if isinstance(retrieved_context, str):
        if retrieved_context.startswith("['") and retrieved_context.endswith("']"):
            try:
                parsed_list = ast.literal_eval(retrieved_context)
                if isinstance(parsed_list, list) and len(parsed_list) > 0:
                    retrieved_context = parsed_list[0]
            except Exception:
                retrieved_context = retrieved_context.strip("[]'\"").replace("\\'", "'")
        
        retrieved_context = retrieved_context.replace('\\n', '\n').replace('""', '"').strip()
        expected_truth = expected_truth.strip()
        
        # Unwrap placeholders for both retrieved contexts and references
        retrieved_context = unwrap_placeholders(retrieved_context)
        expected_truth = unwrap_placeholders(expected_truth)
        
        # Clean double spaces
        retrieved_context = re.sub(r'\s+', ' ', retrieved_context).strip()
        expected_truth = re.sub(r'\s+', ' ', expected_truth).strip()

    # Append to lists
    questions.append(query)
    contexts.append([retrieved_context])
    answers.append(generated_response)  # Collect your final LLM answer
    ground_truths.append(expected_truth)


# ==========================================
# Step 2: Batch Ragas Generation Evaluation
# ==========================================
print(f"\nInitiating end-to-end generation evaluation...")

# Construct the full evaluation dataset
# Ragas expects 'answer' to represent the generated response from your system
eval_data = {
    "question": questions,
    "contexts": contexts,
    "answer": answers,
    "ground_truth": ground_truths
}
ragas_dataset = Dataset.from_dict(eval_data)

# Configure the run to use 2 parallel workers
config = RunConfig(max_workers=2, timeout=60.0)

try:
    # Run evaluation comparing your LLM answers to the ground truths
    results = evaluate(
        dataset=ragas_dataset,
        metrics=[answer_correctness, answer_similarity],
        run_config=config,
        llm=evaluator_llm
    )
    
    # --- FINAL RESULTS DISPLAY ---
    
    print("\n=== OVERALL GENERATION AVERAGE SCORES ===")
    print(results)

    # Convert the results to a Pandas DataFrame
    df_results = results.to_pandas()
    
    print("\n=== DETAILED GENERATION BREAKDOWN ===")
    # Print matching columns for generation metrics in Ragas v0.2
    display(df_results[['user_input', 'response', 'reference', 'answer_correctness', 'answer_similarity']])

except Exception as e:
    print(f"\nGeneration evaluation failed: {e}")

C:\Users\vncpyy7h\AppData\Local\Temp\ipykernel_4196\1400776943.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0.0))


Loading banking dataset from Hugging Face...
Starting end-to-end generation process for 100 samples...
[1/100] Generating response for: 'i cant find my accfount fees i need help checking ...'
[2/100] Generating response for: 'I have made a mistake and I transferred money to t...'
[3/100] Generating response for: 'can I see the loan payment?...'
[4/100] Generating response for: 'i have to see the mortgage payments how could i do...'
[5/100] Generating response for: 'i cannot pay for my loan help me canceling it...'
[6/100] Generating response for: 'I need help to see the current balance on my cards...'
[7/100] Generating response for: 'I want to send money, ia it possible to perform a ...'
[8/100] Generating response for: 'wanna check the  fucking card annual fee i need as...'
[9/100] Generating response for: 'i dont have a fucking profile can ya help me creat...'
[10/100] Generating response for: 'I'm looking for a mortgaeg...'
[11/100] Generating response for: 'i got to check my fees 

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]


=== OVERALL GENERATION AVERAGE SCORES ===
{'answer_correctness': 0.9423, 'answer_similarity': 0.9764}

=== DETAILED GENERATION BREAKDOWN ===


,user_input,response,reference,answer_correctness,answer_similarity
0,i cant find my accfount fees i need help checking them,"I'm genuinely sorry for the inconvenience you're experiencing in locating your account fees. I'm here to help you with that. To check your account fees, please follow these steps:\n\n1. Log in to your account on our website.\n2. Navigate to the ""Account"" or ""Profile"" section.\n3. Look for an option like ""View Account Fees"" or ""Billing Information"".\n4. Click on that option to access detailed information about your account fees.\n\nIf you're unable to locate the account fees or need further assistance, please provide me with your account details or any other relevant information, and I'll gladly assist you further.","I'm genuinely sorry for the inconvenience you're experiencing in locating your account fees. I'm here to help you with that. To check your account fees, please follow these steps: 1. Log in to your account on our website. 2. Navigate to the ""Account"" or ""Profile"" section. 3. Look for an option like ""View Account Fees"" or ""Billing Information"". 4. Click on that option to access detailed information about your account fees. If you're unable to locate the account fees or need further assistance, please provide me with your account details or any other relevant information, and I'll gladly assist you further.",0.999025,0.996101
1,"I have made a mistake and I transferred money to the wrong account, could you help me cancelling a transfer?","1. Please accept my apologies for any inconvenience caused by the mistaken bank transfer. I understand how important it is to resolve this issue for you. Rest assured, I'm here to help you cancel the transfer to the wrong account.\n\n2. To proceed with canceling the transfer, we'll need some specific details. Please provide me with the following information:\n - Your {{Name}}: So I can verify your account details.\n - Date and time of the transfer: This will assist in locating the transaction.\n - The amount and currency transferred: This helps identify the specific transaction among others.\n\n3. Once I have these details, I'll escalate the issue to our support team, who will take the necessary steps to cancel the transfer. Please note that depending on the circumstances, cancellation may not always be guaranteed. However, we'll do everything we can to assist you.\n\n4. If you have any additional questions or concerns, feel free to let me know. We're here to provide you with the support you need.","Please accept my apologies for any inconvenience caused by the mistaken bank transfer. I understand how important it is to resolve this issue for you. Rest assured, I'm here to help you cancel the transfer to the wrong account. To proceed with canceling the transfer, we'll need some specific details. Please provide me with the following information: 1. Your Name: So I can verify your account details. 2. Date and time of the transfer: This will assist in locating the transaction. 3. The amount and currency transferred: This helps identify the specific transaction among others. Once I have these details, I'll escalate the issue to our support team, who will take the necessary steps to cancel the transfer. Please note that depending on the circumstances, cancellation may not always be guaranteed. However, we'll do everything we can to assist you. If you have any additional questions or concerns, feel free to let me know. We're here to provide you with the support you need.",0.957324,0.972153
2,can I see the loan payment?,"1. Log in to your account on our website or mobile app.\n2. Navigate to the ""Loan"" or ""Account"" section.\n3. You should be able to find the information you need regarding your loan payment details.\n\nIf you encounter any issues or have trouble locating the payment details, please let me know, and I'll guide you through the process.","I'd be happy to assist you in checking your loan payment. To view your loan payment details, you 

In [19]:
df_results[['user_input', 'response', 'reference', 'answer_correctness', 'answer_similarity']].sort_values(by='answer_correctness').head(20)

,user_input,response,reference,answer_correctness,answer_similarity
91,"I have to locfk a card, will you help me?","1. Contact our {{Customer Support Team}} immediately, either through our website or by calling {{Customer Support Phone Number}}.\n2. Provide the agent with your card details, such as the card number or account information, to verify your identity.\n3. Explain that you want to block the card and the reason behind it, such as suspected fraudulent activity or loss of the card.\n4. The customer support representative will assist you further and initiate the blocking process.\n5. If necessary, they will also guide you on the next steps, such as ordering a replacement card or disputing any unauthorized transactions.\n\nRemember, time is of the essence when it comes to card security, so please don't delay in reaching out to our customer support team. They are available {{Customer Support Working Hours}} to help you promptly and ensure the safety of your account.","I'm here to assist you with blocking your card. I understand the importance of this matter and I'll guide you through the process. To block your card, please follow these steps: 1. Contact our Customer Support Team immediately, either through our website or by calling Customer Support Phone Number. 2. Provide the agent with your card details, such as the card number or account information, to verify your identity. 3. Explain that you want to block the card and the reason behind it, such as suspected fraudulent activity or loss of the card. 4. The customer support representative will assist you further and initiate the blocking process. 5. If necessary, they will also guide you on the next steps, such as ordering a replacement card or disputing any unauthorized transactions. Remember, time is of the essence when it comes to card security, so please don't delay in reaching out to our customer support team. They are available Customer Support Working Hours to help you promptly and ensure the safety of your account. Rest assured, we're here to support you in any way we can. If you have any additional questions or concerns, feel free to let me know.",0.763757,0.968072
94,"I am looking for information about the nearest branch, where can I get it?","1. Visit our website at {{Company Website URL}}. On the homepage, look for a section called ""Find a Branch"" or ""Branch Locator."" Click on it, and you'll be able to search for the nearest branch by entering your location or ZIP code. The website will provide you with the branch's address, contact information, and even directions if needed.\n\n2. If you prefer speaking to a live representative, you can reach out to our friendly customer support team at {{Customer Support Phone Number}}. They will gladly assist you in finding the nearest branch and provide all the necessary information you're looking for.","Finding information about the nearest branch is crucial, and I'm here to guide you. To get the details you need, you have a couple of options: 1. Visit our website at Company Website URL. On the homepage, look for a section called ""Find a Branch"" or ""Branch Locator."" Click on it, and you'll be able to search for the nearest branch by entering your location or ZIP code. The website will provide you with the branch's address, contact information, and even directions if needed. 2. If you prefer speaking to a live representative, you can reach out to our friendly customer support team at Customer Support Phone Number. They will gladly assist you in finding the nearest branch and provide all the necessary information you're looking for. Remember, we have a dedicated team ready to help you at any time, so don't hesitate to reach out if you have any further questions or need assistance.",0.773390,0.975911
44,i got to check the loan payment how do i do it,"1. Log in to your account on our website.\n2. Navigate to the ""Loan"" or ""My Account"" section.\n3. Look for the option to view your loan details or payments.\n4. Cl